## 13

In [11]:
import torch

def f(x, y):
    return torch.sin((x ** 2) * y)

x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)

result = f(x, y)

result.backward()

print(f"Gradient Vector of f: [{x.grad.item(), y.grad.item()}]",)

Gradient Vector of f: [(1.489864706993103, 0.26291730999946594)]


## 14

In [12]:
import torch.nn as nn

class Dense(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.relu = nn.ReLU()

    def forward(self, X):
        return self.relu(self.linear(X))

In [13]:
torch.manual_seed(42)

dense = Dense(3, 5)
X = torch.randn(5, 3)

y_pred = dense(X)
y_pred.shape

torch.Size([5, 5])

In [14]:
y_pred_check = dense.relu(X @ dense.linear.weight.T + dense.linear.bias)
torch.allclose(y_pred, y_pred_check)

True

In [19]:
import torch.nn.functional as F

class Dense2(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, X):
        z = X @ self.weight.T + self.bias
        return F.relu(z)

In [20]:
torch.manual_seed(42)

dense2 = Dense2(3, 5)
X = torch.randn(5, 3)

y_pred2 = dense2(X)
y_pred2.shape

torch.Size([5, 5])

In [21]:
y_pred_check2 = F.relu(X @ dense2.weight.T + dense2.bias)
torch.allclose(y_pred2, y_pred_check2)

True

## 15

In [93]:
from sklearn.datasets import fetch_covtype
from torch.utils.data import TensorDataset

covtype = fetch_covtype()

X_covtype = torch.tensor(covtype.data, dtype=torch.float32)
means = X_covtype.mean(dim=0, keepdim=True)
stds = X_covtype.std(dim=0, keepdim=True)
X_standardized_covtype = (X_covtype - means) / stds

y_covtype = torch.tensor(covtype.target - 1, dtype=torch.long)

covtype_dataset = TensorDataset(X_standardized_covtype, y_covtype)

In [94]:
sample0, target0 = covtype_dataset[0]
sample0.shape, target0.shape

(torch.Size([54]), torch.Size([]))

In [95]:
from torch.utils.data import random_split

torch.manual_seed(42)

train_size = len(covtype_dataset) * 80 // 100
valid_size = len(covtype_dataset) * 10 // 100
test_size = len(covtype_dataset) - train_size - valid_size

train_dataset, valid_dataset, test_dataset = random_split(
    covtype_dataset, 
    [train_size, valid_size, test_size]
)

In [96]:
from torch.utils.data import DataLoader

batch_size = 512

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [97]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [98]:
n_inputs = len(covtype.feature_names)
n_classes = len(set(covtype.target))

torch.manual_seed(42)
covtype_model = nn.Sequential(
    nn.Linear(n_inputs, 200), nn.ReLU(),
    nn.Linear(200, 100), nn.ReLU(),
    nn.Linear(100, 50), nn.ReLU(),
    nn.Linear(50, n_classes)
).to(device)

In [99]:
class CoverTypeModel(nn.Module):
    def __init__(self, n_neurons, n_inputs=n_inputs, n_classes=n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_inputs, n_neurons[0]),
            nn.ReLU(),
            nn.Linear(n_neurons[0], n_neurons[1]),
            nn.ReLU(),
            nn.Linear(n_neurons[1], n_neurons[2]),
            nn.ReLU(),
            nn.Linear(n_neurons[2], n_classes)
        )

    def forward(self, X):
        return self.mlp(X)
    
torch.manual_seed(42)
covtype_model = CoverTypeModel([200, 100, 50]).to(device)

In [100]:
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}

    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()

        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate(model, valid_loader, metric).item())
        
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

In [101]:
import torchmetrics

for learning_rate in [0.16, 0.08, 0.04, 0.02, 0.01]:
    n_epochs = 15
    optimizer = torch.optim.SGD(covtype_model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    metric = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)

    history = train(covtype_model, optimizer, criterion, metric, 
                    train_loader, valid_loader, n_epochs)

Epoch 1/15, train loss: 0.6900, train metric: 0.7148, valid metric: 0.7524
Epoch 2/15, train loss: 0.5594, train metric: 0.7614, valid metric: 0.7757
Epoch 3/15, train loss: 0.5177, train metric: 0.7766, valid metric: 0.7871
Epoch 4/15, train loss: 0.4852, train metric: 0.7917, valid metric: 0.8082
Epoch 5/15, train loss: 0.4556, train metric: 0.8058, valid metric: 0.8161
Epoch 6/15, train loss: 0.4316, train metric: 0.8167, valid metric: 0.8264
Epoch 7/15, train loss: 0.4100, train metric: 0.8269, valid metric: 0.8388
Epoch 8/15, train loss: 0.3899, train metric: 0.8364, valid metric: 0.8517
Epoch 9/15, train loss: 0.3714, train metric: 0.8451, valid metric: 0.8567
Epoch 10/15, train loss: 0.3565, train metric: 0.8525, valid metric: 0.8655
Epoch 11/15, train loss: 0.3441, train metric: 0.8577, valid metric: 0.8667
Epoch 12/15, train loss: 0.3288, train metric: 0.8639, valid metric: 0.8618
Epoch 13/15, train loss: 0.3185, train metric: 0.8687, valid metric: 0.8726
Epoch 14/15, train lo

In [102]:
evaluate(covtype_model, test_loader, metric)

tensor(0.9380, device='cuda:0')

In [103]:
import optuna

def objective(trail):
    learning_rate = trail.suggest_float("learning_rate", 1e-2, 1.0, log=True)
    n_hidden_1 = trail.suggest_int("n_hidden_1", 30, 200)
    n_hidden_2 = trail.suggest_int("n_hidden_2", 30, 200)
    n_hidden_3 = trail.suggest_int("n_hidden_3", 30, 200)
    
    covtype_model = CoverTypeModel([n_hidden_1, n_hidden_2, n_hidden_3], n_inputs, n_classes).to(device)
    optimizer = torch.optim.SGD(covtype_model.parameters(), lr=learning_rate)

    xentropy = nn.CrossEntropyLoss()
    metric = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)
    n_epochs = 20

    for epoch in range(n_epochs):
        covtype_model.train()
        metric.reset()
        total_loss = 0.

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = covtype_model(X_batch)
            loss = xentropy(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            
        mean_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()
        valid_metric = evaluate(covtype_model, valid_loader, metric).item()
        
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {mean_loss:.4f}, "
              f"train metric: {train_metric:.4f}, "
              f"valid metric: {valid_metric:.4f}")

        trail.report(valid_metric, epoch)
        if trail.should_prune():
            raise optuna.TrialPruned()
    
    return valid_metric

In [104]:
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=1, n_warmup_steps=3, interval_steps=1)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=5)

[I 2026-07-13 23:35:07,395] A new study created in memory with name: no-name-66e10d64-d986-4bc7-93d5-84e32a63421a


Epoch 1/20, train loss: 0.7923, train metric: 0.6825, valid metric: 0.7379
Epoch 2/20, train loss: 0.6105, train metric: 0.7425, valid metric: 0.7574
Epoch 3/20, train loss: 0.5705, train metric: 0.7567, valid metric: 0.7643
Epoch 4/20, train loss: 0.5447, train metric: 0.7663, valid metric: 0.7703
Epoch 5/20, train loss: 0.5256, train metric: 0.7745, valid metric: 0.7886
Epoch 6/20, train loss: 0.5048, train metric: 0.7835, valid metric: 0.7969
Epoch 7/20, train loss: 0.4902, train metric: 0.7896, valid metric: 0.7958
Epoch 8/20, train loss: 0.4725, train metric: 0.7979, valid metric: 0.8107
Epoch 9/20, train loss: 0.4611, train metric: 0.8027, valid metric: 0.8018
Epoch 10/20, train loss: 0.4491, train metric: 0.8076, valid metric: 0.8196
Epoch 11/20, train loss: 0.4344, train metric: 0.8158, valid metric: 0.8255
Epoch 12/20, train loss: 0.4275, train metric: 0.8180, valid metric: 0.8305
Epoch 13/20, train loss: 0.4162, train metric: 0.8233, valid metric: 0.8276
Epoch 14/20, train lo

[I 2026-07-13 23:40:15,915] Trial 0 finished with value: 0.8599335551261902 and parameters: {'learning_rate': 0.05611516415334506, 'n_hidden_1': 192, 'n_hidden_2': 155, 'n_hidden_3': 132}. Best is trial 0 with value: 0.8599335551261902.


Epoch 20/20, train loss: 0.3600, train metric: 0.8494, valid metric: 0.8599
Epoch 1/20, train loss: 1.0856, train metric: 0.5479, valid metric: 0.6753
Epoch 2/20, train loss: 0.7204, train metric: 0.7084, valid metric: 0.7266
Epoch 3/20, train loss: 0.6636, train metric: 0.7294, valid metric: 0.7370


[I 2026-07-13 23:41:14,680] Trial 1 pruned. 


...

Epoch 20/20, train loss: 0.2913, train metric: 0.8806, valid metric: 0.8877
Epoch 1/20, train loss: 0.6439, train metric: 0.7259, valid metric: 0.7578
Epoch 2/20, train loss: 0.5177, train metric: 0.7743, valid metric: 0.7956
Epoch 3/20, train loss: 0.4642, train metric: 0.7998, valid metric: 0.8161
Epoch 4/20, train loss: 0.4282, train metric: 0.8174, valid metric: 0.8194
Epoch 5/20, train loss: 0.3963, train metric: 0.8329, valid metric: 0.8231
Epoch 6/20, train loss: 0.3716, train metric: 0.8444, valid metric: 0.8534
Epoch 7/20, train loss: 0.3528, train metric: 0.8537, valid metric: 0.8518
Epoch 8/20, train loss: 0.3366, train metric: 0.8604, valid metric: 0.8663
Epoch 9/20, train loss: 0.3226, train metric: 0.8673, valid metric: 0.8753
Epoch 10/20, train loss: 0.3114, train metric: 0.8721, valid metric: 0.8698
Epoch 11/20, train loss: 0.3023, train metric: 0.8764, valid metric: 0.8756
Epoch 12/20, train loss: 0.2945, train metric: 0.8797, valid metric: 0.8861
Epoch 13/20, train lo

[I 2026-07-13 23:52:21,426] Trial 3 finished with value: 0.8991755843162537 and parameters: {'learning_rate': 0.46225890010208287, 'n_hidden_1': 66, 'n_hidden_2': 61, 'n_hidden_3': 61}. Best is trial 3 with value: 0.8991755843162537.


Epoch 20/20, train loss: 0.2540, train metric: 0.8965, valid metric: 0.8992
Epoch 1/20, train loss: 0.8831, train metric: 0.6397, valid metric: 0.7341
Epoch 2/20, train loss: 0.6433, train metric: 0.7366, valid metric: 0.7433
Epoch 3/20, train loss: 0.6017, train metric: 0.7469, valid metric: 0.7552


[I 2026-07-13 23:53:48,833] Trial 4 pruned. 


Epoch 4/20, train loss: 0.5715, train metric: 0.7581, valid metric: 0.7629


In [105]:
print(study.best_params)
print(study.best_value)

{'learning_rate': 0.46225890010208287, 'n_hidden_1': 66, 'n_hidden_2': 61, 'n_hidden_3': 61}
0.8991755843162537
